# Conference paper comparisons — FashionIQ / Colab T4

Use one GPU task at a time. Downloads and reporting can run on a CPU runtime. Completed UniFashion training is reused; the teammate's integrated gate remains pending.

Upload `checkpoints/paper_comparisons.zip` to your Week8 `inputs` folder. Reuse the existing `unifashion_week8_data.zip` and `pretrained_assets`. See `docs/Conference_Comparisons_Runbook.md` for protocol differences and limits. No result is claimed until all 6,016 validation queries complete.


## Cell 1 — Setup and mount Drive

Rerun this and Cell 2 after a runtime reset. Only the new workspace `/content/fashioniq_paper` is used. Persistent results remain in Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, hashlib, shutil, subprocess, zipfile
from pathlib import Path

WEEK8 = Path('/content/drive/MyDrive/Thesis/UniFashion/week8')
PAPER = Path('/content/drive/MyDrive/Thesis/UniFashion/paper_comparisons')
WORK = Path('/content/fashioniq_paper')
BUNDLE = WEEK8 / 'inputs/paper_comparisons.zip'
DATA_ZIP = WEEK8 / 'inputs/unifashion_week8_data.zip'
ORIGINAL_ASSETS = WEEK8 / 'pretrained_assets'
ASSETS, FEATURES, RESULTS, REPORT = (PAPER / p for p in ('assets', 'features', 'results', 'report'))
for p in (PAPER, WORK, ASSETS, RESULTS, REPORT):
    p.mkdir(parents=True, exist_ok=True)
for p in (BUNDLE, DATA_ZIP, ORIGINAL_ASSETS / 'download_manifest.json'):
    assert p.is_file(), f'Missing required input: {p}'
os.environ['HF_HOME'] = str(ASSETS / 'huggingface')
os.environ['MPLCONFIGDIR'] = str(WORK / '.cache/matplotlib')

def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(8*1024*1024), b''):
            h.update(block)
    return h.hexdigest()

def unpack(archive, destination):
    root = destination.resolve()
    with zipfile.ZipFile(archive) as z:
        for info in z.infolist():
            target = (root / info.filename).resolve()
            if not target.is_relative_to(root):
                raise ValueError('Unsafe ZIP member: ' + info.filename)
        z.extractall(root)

unpack(BUNDLE, WORK)
manifest = json.loads((WORK / 'paper_bundle_manifest.json').read_text())
for name, expected in manifest['files'].items():
    assert sha(WORK / name) == expected, f'Bundle file differs: {name}'
# Images are verified against the audited manifest by the evaluation/extraction scripts.
if not (WORK / 'data/data_audit.json').exists():
    unpack(DATA_ZIP, WORK)
old_gate = WORK / 'logs/paper_comparisons/legacy_gate'
if not (RESULTS / 'legacy_gate').exists():
    shutil.copytree(old_gate, RESULTS / 'legacy_gate')
os.chdir(WORK)

def run_live(command):
    command = [str(x) for x in command]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1, cwd=WORK)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait():
        raise RuntimeError(f'Command failed ({process.returncode}); see the output above.')

print('Persistent comparison folder:', PAPER)
print('Bundle SHA256:', sha(BUNDLE))


## Cell 2 — Compatible dependencies and CPU checks

This keeps Colab's torch/torchvision pair. Every ML operation runs in a new Python subprocess, so notebook-kernel imports cannot retain the old pandas/Transformers installation. The small adapter tests do not load the full checkpoints.


In [ ]:
packages = [
    'numpy==1.26.4', 'pandas==2.2.3', 'transformers==4.36.2', 'tokenizers==0.15.2',
    'timm==0.9.12', 'omegaconf==2.3.0', 'iopath==0.1.10', 'fairscale==0.4.13',
    'einops==0.7.0', 'accelerate==0.27.2', 'Pillow==11.1.0', 'matplotlib==3.9.4',
    'pytest==8.3.5', 'gdown==5.2.0', 'ftfy==6.3.1', 'regex', 'safetensors'
]
run_live([sys.executable, '-m', 'pip', 'install', '-q'] + packages)
clip_revision = json.loads(Path('configs/paper_comparisons.json').read_text())['openai_clip']['revision']
run_live([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
          'git+https://github.com/openai/CLIP.git@' + clip_revision])
run_live([sys.executable, '-B', '-m', 'pytest', 'tests/test_paper_comparisons.py',
          '-q', '--junitxml', RESULTS / 'tests_colab.xml'])


## Cell 3 — Download the selected assets (CPU is sufficient)

Choose `fashionclip` first, then prepare `sprc` and `clip4cir` when needed. Change runtime to T4 only for Cell 4. CLIP4CIR downloads 8.5 GB of split release files to local disk and keeps just the selected FashionIQ model pair in Drive. A runtime reset during that download may lose unfinished local parts. SPRC downloads use the official Drive link and may be subject to host quotas.


In [ ]:
DOWNLOAD_METHOD = 'fashionclip'  # 'fashionclip', 'sprc', or 'clip4cir'
assert DOWNLOAD_METHOD in ('fashionclip', 'sprc', 'clip4cir')
run_live([sys.executable, '-B', '-u', 'scripts/prepare_paper_models.py',
          '--model', DOWNLOAD_METHOD, '--output', ASSETS,
          '--download-cache', Path('/content/paper_download_parts')])


## Cell 4 — Run one GPU task, or resume it

Recommended order: `features_controls`, `sprc`, `clip4cir`, `unifashion-initial`. Prepare each method's assets in Cell 3 first. Keep `BATCH_SIZE = 4` for resumable external evaluations. `features_controls` uses 32 for feature extraction and 256 for the small frozen-feature training; it runs seeds 42, 43 and 44 for all three control models.

Checkpoints/chunks are saved to Drive. If usage runs out, rerun Cells 1–2 and this cell with the same `TASK`. A completed chunk/epoch is reused. Changed input/code/version checks require a new experiment folder.


In [ ]:
TASK = 'features_controls'  # 'sprc', 'clip4cir', or 'unifashion-initial'
BATCH_SIZE = 4
assert TASK in ('features_controls', 'sprc', 'clip4cir', 'unifashion-initial')
run_live([sys.executable, '-c',
          "import torch; assert torch.cuda.is_available(), 'Select a GPU runtime'; print(torch.cuda.get_device_name(), torch.__version__)"])
EVAL_DATA = WORK / 'logs/week41_fashioniq_evaluation/evaluation_data'
REFERENCE = WORK / 'logs/week43_44_unifashion_completion/evidence_completed_20261102/final_results/unifashion_image_score.json'

if TASK == 'features_controls':
    run_live([sys.executable, '-B', '-u', 'scripts/extract_paper_fashionclip.py',
              '--data-dir', WORK / 'data', '--images-dir', WORK / 'images',
              '--eval-data', EVAL_DATA, '--metadata', WORK / 'logs/fashionclip_text_metadata.csv',
              '--output', FEATURES, '--batch-size', '32'])
    run_live([sys.executable, '-B', '-u', 'scripts/run_fashionclip_controls.py',
              '--embeddings', FEATURES, '--eval-data', EVAL_DATA,
              '--output', RESULTS / 'matched_controls', '--device', 'cuda',
              '--epochs', '15', '--batch-size', '256', '--seeds', '42', '43', '44'])
else:
    run_live([sys.executable, '-B', '-u', 'scripts/evaluate_paper_models.py',
              '--model', TASK, '--assets', ASSETS,
              '--data-dir', WORK / 'data', '--images-dir', WORK / 'images',
              '--eval-data', EVAL_DATA, '--output', RESULTS / TASK,
              '--original-code', WORK / 'checkpoints/unifashion_week8_code.zip',
              '--unifashion-assets', ORIGINAL_ASSETS, '--upstream', WORK / 'third_party/UniFashion',
              '--reference-result', REFERENCE, '--batch-size', str(BATCH_SIZE)])
print('Task finished:', TASK)


## Cell 5 — Rebuild the paper comparison report

Run after any completed task, including after a session with partial progress. Incomplete methods remain pending and are excluded from result tables. Published paper numbers remain separate from measured results.


In [ ]:
run_live([sys.executable, '-B', 'scripts/report_paper_comparisons.py',
          '--root', WORK, '--new-results', RESULTS, '--output', REPORT])
from IPython.display import display, Markdown
display(Markdown((REPORT / 'comparison_report.md').read_text()))


## Cell 6 — Export the evidence for review

This excludes model checkpoints, embeddings, images and resumable caches. Keep those files in Drive to continue later. Send `paper_comparison_evidence.zip` with the executed notebook; do not call the integrated UniFashion gate comparison complete until your teammate supplies its checkpoint.


In [ ]:
run_live([sys.executable, '-B', 'scripts/export_paper_comparisons.py',
          '--results', RESULTS, '--output', PAPER / 'run_evidence.zip'])
# Include reports alongside lightweight run evidence.
EXPORT = PAPER / 'paper_comparison_evidence.zip'
with zipfile.ZipFile(PAPER / 'run_evidence.zip') as source, zipfile.ZipFile(EXPORT, 'w', zipfile.ZIP_DEFLATED) as target:
    for name in source.namelist():
        target.writestr('results/' + name, source.read(name))
    for path in REPORT.iterdir():
        if path.is_file() and path.suffix in ('.json', '.csv', '.md', '.tex', '.png', '.svg'):
            target.write(path, 'report/' + path.name)
print('Evidence saved:', EXPORT)
from google.colab import files
files.download(str(EXPORT))
